In [1]:
"""
HyDE -- Hypothetical Document Embeddings RAG Pipeline -- Production-Grade
==========================================================================
Architecture   : FIVE configurations covering every HyDE variant and production pattern:
                 (1) Standard RAG Baseline          -- direct query embedding, zero hypothetical
                 (2) HyDE via HypotheticalDocumentEmbedder -- built-in LangChain, prompt_key="sci_fact"
                 (3) Multi-Hypothesis HyDE          -- N=5 generations, centroid-averaged embedding
                 (4) Domain-Specific Custom-Prompt HyDE -- research-paper-tuned generation prompt
                 (5) HyDE + CrossEncoder Reranker + HyPE [BEST] -- full production hybrid

Paper          : "Precise Zero-Shot Dense Retrieval without Relevance Labels"
                  Gao et al., NAACL 2022   arXiv:2212.10496

Vector Store   : FAISS  (IndexFlatIP, cosine similarity)
Embeddings     : BAAI/bge-large-en-v1.5  (1024-dim, query instruction prefix)
Reranker       : BAAI/bge-reranker-base  (CrossEncoder, Config 5 only)
LLM            : Azure OpenAI  (ChatOllama -- GPT-4o)
Framework      : LangChain 0.2+  (HypotheticalDocumentEmbedder, LCEL, custom chains)

Reference PDFs (open-access arXiv):
    1. "Attention Is All You Need" -- Vaswani et al., 2017
       https://arxiv.org/pdf/1706.03762.pdf
    2. "BERT: Pre-training of Deep Bidirectional Transformers" -- Devlin et al., 2018
       https://arxiv.org/pdf/1810.04805.pdf
    3. "RAG for Knowledge-Intensive NLP Tasks" -- Lewis et al., 2020
       https://arxiv.org/pdf/2005.11401.pdf

=============================================================================
The Core Problem HyDE Solves: Query-Document Embedding Space Asymmetry
=============================================================================
Standard dense retrieval embeds BOTH the query and each document into the
same high-dimensional vector space and retrieves by cosine similarity.
The fundamental assumption is that a relevant document and its corresponding
query embed to nearby vectors. This assumption holds well when query and
document are stylistically similar -- but systematically breaks in practice:

    QUERY STYLE:  Short (5-15 tokens). Interrogative. No context.
                  Sparse vocabulary. Often imperative or keyword-like.
                  Example: "BERT pre-training objectives?"

    DOCUMENT STYLE: Long (50-200 tokens). Declarative. Rich context.
                    Dense technical vocabulary. Expository prose.
                    Example: "BERT uses two unsupervised prediction tasks
                    for pre-training: Masked Language Model (MLM), where
                    15% of tokens are randomly masked and the model must
                    predict them, and Next Sentence Prediction (NSP)..."

These two pieces of text, though semantically matched, embed to vectors that
are often 0.3-0.5 cosine distance apart -- far enough to miss the top-K
cutoff when competing against document chunks that happen to share surface
vocabulary with the query but are semantically less relevant.

WHY THE ASYMMETRY EXISTS:
    Embedding models like BGE-large learn to represent text in a unified space.
    But the training signal for queries is typically from query-answer pairs
    (MS MARCO, NQ, TriviaQA), while the document side is trained on passage-
    level contrastive objectives. The two distributions occupy overlapping but
    distinct regions of the embedding manifold.
    A 5-token query vector sits in a different part of the manifold than
    a 100-token passage vector, even when they are about the same concept.

HOW HYDE SOLVES IT:
    Instead of embedding the query directly, HyDE:
        1. Generates a HYPOTHETICAL DOCUMENT -- a plausible, fake answer
           passage written in the SAME STYLE as the indexed documents
           (declarative, dense, expository).
        2. Embeds this hypothetical document instead of the query.
        3. Performs cosine similarity search between the hypothetical
           document embedding and the indexed passage embeddings.
    This transforms the search from QUERY-TO-DOCUMENT similarity
    (cross-style, high variance) to DOCUMENT-TO-DOCUMENT similarity
    (same-style, lower variance), dramatically improving recall.

    The hypothetical document does NOT need to be factually correct.
    It only needs to OCCUPY THE SAME REGION OF EMBEDDING SPACE as the
    true answer documents. Gao et al. (2022) show that even hallucinated
    hypothetical answers retrieve the correct passages, because the
    structural and vocabulary patterns of the hypothetical match the
    real documents far better than the raw query does.

=============================================================================
The Embedding Space Geometry: Why Doc-to-Doc Works
=============================================================================
Consider BGE-large's 1024-dimensional embedding space.
Define three points:
    q = embed("What are BERT's pre-training objectives?")          -- query
    h = embed("BERT is pre-trained with MLM and NSP objectives")  -- hypothetical
    d = embed("BERT uses masked language model and NSP for pre-training...")  -- real doc

Typical distances in this space:
    cosine(q, d) ~ 0.72   (query-to-document, cross-style)
    cosine(h, d) ~ 0.91   (hypothetical-to-document, same-style)
    cosine(q, h) ~ 0.85   (query-to-hypothetical, benefiting from hypothesis)

The hypothetical h acts as a QUERY AMPLIFIER: it moves the search vector
from the query manifold into the document manifold, where the indexed
passages reside. The 0.19 improvement in cosine similarity translates to
significantly better rank position in top-K retrieval.

=============================================================================
Multi-Hypothesis Averaging: Reducing Variance via Centroid
=============================================================================
A single LLM generation is stochastic (temperature > 0) and may contain
hallucinations that pull the embedding away from the true answer region.
Multi-hypothesis HyDE generates N independent hypothetical documents and
averages their embedding vectors:

    h_centroid = (1/N) * sum_{i=1}^{N} embed(hypothetical_i)

This centroid has two desirable properties:
    1. VARIANCE REDUCTION: Each hypothesis has a different hallucinated detail.
       Averaging cancels out the idiosyncratic hallucination directions,
       leaving only the signal that is consistent across all N hypotheses --
       precisely the core semantic content of the correct answer.
    2. COVERAGE EXPANSION: Different hypotheses may emphasize different aspects
       of the answer ("MLM training" vs "masked token prediction").
       The centroid covers a broader region of the document manifold,
       increasing the chance that at least one relevant passage is in top-K.

The original HyDE paper uses N=8 hypotheses. In production, N=3-5 provides
most of the benefit at reduced LLM cost (N LLM calls per query).

=============================================================================
HyPE -- Hypothetical Prompt Embeddings (2025 Index-Time Variant)
=============================================================================
HyDE operates at QUERY TIME: each query triggers one or more LLM generations.
HyPE (Vake et al., 2025) inverts this: it operates at INDEX TIME.

    HyDE:  query -> LLM -> hypothetical_doc -> embed -> search (runtime cost)
    HyPE:  chunk -> LLM -> hypothetical_questions -> embed -> index (offline cost)

During indexing, for each document chunk, HyPE generates K hypothetical
questions that the chunk could answer, embeds them, and stores those
embeddings alongside the chunk. At query time, the raw query is embedded
and matched against the QUESTION embeddings (same style: question-to-question).

    HyPE reported gains (Vake et al., 2025):
        +42 percentage points precision on select BEIR datasets
        +45 percentage points recall on select BEIR datasets
        ZERO query-time generation cost (only offline indexing cost)

In Config 5, we implement both HyDE (query-time) and HyPE (index-time)
and combine them: HyPE provides high-precision recall at zero query latency,
and a light single-hypothesis HyDE handles queries outside the HyPE coverage.

"""

'\nHyDE -- Hypothetical Document Embeddings RAG Pipeline -- Production-Grade\n==========================================================================\nArchitecture   : FIVE configurations covering every HyDE variant and production pattern:\n                 (1) Standard RAG Baseline          -- direct query embedding, zero hypothetical\n                 (2) HyDE via HypotheticalDocumentEmbedder -- built-in LangChain, prompt_key="sci_fact"\n                 (3) Multi-Hypothesis HyDE          -- N=5 generations, centroid-averaged embedding\n                 (4) Domain-Specific Custom-Prompt HyDE -- research-paper-tuned generation prompt\n                 (5) HyDE + CrossEncoder Reranker + HyPE [BEST] -- full production hybrid\n\nPaper          : "Precise Zero-Shot Dense Retrieval without Relevance Labels"\n                  Gao et al., NAACL 2022   arXiv:2212.10496\n\nVector Store   : FAISS  (IndexFlatIP, cosine similarity)\nEmbeddings     : BAAI/bge-large-en-v1.5  (1024-dim, query in

In [2]:
import os 
import sys 
import time 
import logging 
import urllib.request 
from dataclasses import dataclass , field 
from pathlib import Path 
from typing import List,Dict,Tuple,Optional,Any 
import numpy as np 
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS 
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate 
from langchain_core.runnables import RunnableLambda,RunnablePassthrough
from langchain_ollama import ChatOllama
from langchain_community.embeddings import HypotheticalDocumentEmbedder
from langchain_core.output_parsers import StrOutputParser


C:\Users\dhanu\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0520 21:16:31.148000 59608 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [3]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("hyde_rag")


In [22]:

class HyDEConfig:
    """
    Centralized configuration for the HyDE RAG pipeline.

    HyDE-Specific Parameter Decisions:

    HYDE_TEMPERATURE (0.5):
        The hypothetical document generation requires CONTROLLED CREATIVITY.
        Temperature=0.0 produces identical or near-identical hypotheticals
        on repeated calls -- no benefit to generating multiple hypotheses.
        Temperature=1.0 produces highly variable hypotheticals that may
        diverge far from the true answer region.
        Temperature=0.5 is the validated production default from the HyDE
        paper: enough variation for multi-hypothesis coverage, not so much
        that individual hypotheses embed far from the target region.
        Use 0.3-0.4 for technical/factual corpora (lower hallucination risk).
        Use 0.6-0.7 for creative/open-ended corpora.

    N_HYPOTHESES (5):
        Number of independent hypothetical documents generated per query
        for multi-hypothesis averaging (Config 3).
        N=1: single hypothesis, no averaging benefit.
        N=3: good balance of cost vs. variance reduction.
        N=5: strong variance reduction, 5x query-time LLM cost.
        N=8: original paper default, full benefit, 8x cost.
        Production recommendation: N=3 for latency-sensitive, N=5 for quality.

    HYDE_MAX_TOKENS (256):
        Maximum tokens in the generated hypothetical document.
        The hypothetical should be approximately the SAME LENGTH as the
        indexed chunks (CHUNK_SIZE=450 chars ~ 65-75 tokens).
        Longer hypotheticals dilute the embedding toward a "summary of everything"
        rather than a targeted answer passage. 256 tokens is the upper bound;
        typical hypotheticals are 80-150 tokens.

    HYPE_N_QUESTIONS (3):
        Number of hypothetical questions generated per chunk during HyPE indexing.
        Each question is embedded and stored as an additional search vector for
        the chunk. A chunk with 3 HyPE question embeddings has 3 entry points
        in the embedding space for retrieval.
        Higher N_QUESTIONS = better coverage = larger index (3x more FAISS vectors).

    LLM_TEMPERATURE vs HYDE_TEMPERATURE:
        LLM_TEMPERATURE=0.0 is used for the ANSWER GENERATION step (final RAG).
        HYDE_TEMPERATURE=0.5 is used only for HYPOTHETICAL DOCUMENT GENERATION.
        These are separate because answer generation requires maximum consistency
        while hypothesis generation benefits from controlled diversity.
    """

    # --- Dataset -----------------------------------------------------------
    PDF_SOURCES: List[Tuple[str, str]] = [
        ("attention_is_all_you_need",    "https://arxiv.org/pdf/1706.03762.pdf"),
        ("bert_pretraining_transformers", "https://arxiv.org/pdf/1810.04805.pdf"),
        ("rag_knowledge_intensive_nlp",  "https://arxiv.org/pdf/2005.11401.pdf"),
    ]
    PDF_DOWNLOAD_DIR: str = "./pdfs"

    # --- FAISS Indices (separate indices for standard and HyPE) -----------
    FAISS_STANDARD_DIR:  str = "./faiss_hyde_standard"
    FAISS_HYPE_DIR:      str = "./faiss_hyde_hype"

    # --- BGE Bi-Encoder ---------------------------------------------------
    BIENCODER_MODEL:             str = "BAAI/bge-large-en-v1.5"
    BIENCODER_DEVICE:            str = "cpu"
    BIENCODER_QUERY_INSTRUCTION: str = (
        "Represent this sentence for searching relevant passages: "
    )

    # --- CrossEncoder Reranker (Config 5) ---------------------------------
    RERANKER_MODEL: str = "BAAI/bge-reranker-base"

    # --- Text Splitting ---------------------------------------------------
    CHUNK_SIZE:    int = 450
    CHUNK_OVERLAP: int = 60

    # --- HyDE Parameters --------------------------------------------------
    HYDE_TEMPERATURE: float = 0.5     # hypothesis generation temperature
    HYDE_MAX_TOKENS:  int   = 256     # max tokens per hypothetical document
    N_HYPOTHESES:     int   = 5       # number of hypotheses for multi-avg (Config 3)
    SEARCH_K:         int   = 4       # final documents returned to LLM
    CANDIDATE_K:      int   = 20      # candidates before reranking (Config 5)

    # --- HyPE Parameters (Config 5, index-time) ---------------------------
    HYPE_N_QUESTIONS: int   = 3       # questions generated per chunk at index time
    HYPE_TEMPERATURE: float = 0.4     # lower temp: more focused questions

    # --- Azure OpenAI LLM (answer generation) ----------------------------
    OLLAMA_BASE_URL: str = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
    OLLAMA_MODEL:    str = os.environ.get("OLLAMA_MODEL",    "qwen2.5-coder:7b")
    LLM_TEMPERATURE:  float        = 0.0     # deterministic for answer generation
    LLM_MAX_TOKENS:   int          = 1024

    # --- RAG Answer Prompt -----------------------------------------------
    RAG_PROMPT: str = """You are a precise technical research assistant.
Answer the question using ONLY the information in the context below.
If the answer is not in the context, respond:
"The provided documents do not contain sufficient information to answer this question."

Context (retrieved using HyDE -- hypothetical document embedding search):
{context}

Question: {question}

Precise technical answer:"""

    # --- HyDE Hypothesis Generation Prompt (Config 4 custom) -------------
    HYDE_RESEARCH_PAPER_PROMPT: str = """You are an expert in deep learning, NLP, and transformer architectures.
Write a detailed, technical research paper excerpt that precisely answers the following question.
Your response must:
- Be written in the style of a machine learning research paper (declarative, precise, technical)
- Include specific technical details, numbers, dimensions, or formulas if relevant
- Be 80 to 150 words in length
- NOT include disclaimers, preambles, or meta-commentary
- Sound like it was extracted directly from an academic paper

Question: {question}

Research paper excerpt (answer the question directly in paper-excerpt style):"""


In [23]:

# ===========================================================================
# SECTION 2 -- HyDE TRACE DATA CLASS
# ===========================================================================

@dataclass
class HyDETrace:
    """
    Captures the complete HyDE execution trace for observability.

    Fields:
        question           : Original user query.
        strategy           : Config label.
        hypotheticals      : List of generated hypothetical documents.
        hyp_embed_norms    : L2 norms of each hypothetical embedding (quality indicator).
        centroid_cosines   : Cosine similarity of each hypothetical to centroid (diversity).
        retrieved_docs     : Final retrieved documents after HyDE search.
        final_answer       : LLM-generated answer from retrieved context.
        hyde_latency_ms    : Time for all hypothesis generations (LLM calls).
        embed_latency_ms   : Time for embedding all hypotheticals.
        retrieval_latency_ms: Time for FAISS similarity search.
        total_ms           : End-to-end wall clock time.

    The hyp_embed_norms and centroid_cosines fields are diagnostic:
        - If all norms are near 1.0: BGE normalize_embeddings=True is working.
        - If centroid_cosines are all above 0.95: hypotheticals are too similar,
          N_HYPOTHESES is not adding diversity (lower temperature needed).
        - If centroid_cosines vary between 0.7-0.9: healthy diversity range.
    """
    question:              str
    strategy:              str
    hypotheticals:         List[str]   = field(default_factory=list)
    hyp_embed_norms:       List[float] = field(default_factory=list)
    centroid_cosines:      List[float] = field(default_factory=list)
    retrieved_docs:        List[Document] = field(default_factory=list)
    final_answer:          str         = ""
    hyde_latency_ms:       float       = 0.0
    embed_latency_ms:      float       = 0.0
    retrieval_latency_ms:  float       = 0.0
    total_ms:              float       = 0.0

    def print_trace(self) -> None:
        """Print human-readable HyDE execution trace."""
        print(f"\n{'='*70}")
        print(f"HYDE TRACE: {self.strategy}")
        print(f"Question: {self.question[:80]}")
        print(f"{'='*70}")

        if self.hypotheticals:
            print(f"\n  Hypothetical Documents Generated ({len(self.hypotheticals)}):")
            for i, hyp in enumerate(self.hypotheticals, 1):
                norm    = self.hyp_embed_norms[i-1] if i <= len(self.hyp_embed_norms) else "?"
                cosine  = self.centroid_cosines[i-1] if i <= len(self.centroid_cosines) else "?"
                cosine_str = f"{cosine:.4f}" if isinstance(cosine, float) else str(cosine)
                norm_str   = f"{norm:.4f}"   if isinstance(norm,   float) else str(norm)
                print(f"\n  [Hyp {i}]  |norm|={norm_str}  cos_to_centroid={cosine_str}")
                print(f"    {hyp[:200].replace(chr(10), ' ')}...")

        if self.retrieved_docs:
            print(f"\n  Retrieved Docs ({len(self.retrieved_docs)}):")
            for i, doc in enumerate(self.retrieved_docs, 1):
                paper = doc.metadata.get("paper_name", "?")
                page  = doc.metadata.get("page", "?")
                chars = len(doc.page_content)
                snip  = doc.page_content[:150].replace("\n", " ")
                print(f"  [Rank {i}] {paper} | Page {page} | {chars} chars")
                print(f"            {snip}...")

        print(f"\n  Latency: HyDE={self.hyde_latency_ms:.1f}ms "
              f"  Embed={self.embed_latency_ms:.1f}ms "
              f"  Search={self.retrieval_latency_ms:.1f}ms "
              f"  Total={self.total_ms:.1f}ms")
        print(f"\n  ANSWER:\n  {self.final_answer[:400]}")
        print("=" * 70 + "\n")


In [24]:

# ===========================================================================
# SECTION 3 -- PDF DOWNLOADER AND CHUNKER
# ===========================================================================

def download_pdfs(config: HyDEConfig) -> List[Path]:
    """Download research PDFs with file-existence caching."""
    dl_dir = Path(config.PDF_DOWNLOAD_DIR)
    dl_dir.mkdir(parents=True, exist_ok=True)
    paths: List[Path] = []

    for name, url in config.PDF_SOURCES:
        dest = dl_dir / f"{name}.pdf"
        if dest.exists() and dest.stat().st_size > 10_000:
            logger.info("Cached: %s  (%.1f KB)", dest.name, dest.stat().st_size / 1024)
            paths.append(dest)
            continue

        logger.info("Downloading: %s", url)
        try:
            req = urllib.request.Request(
                url, headers={"User-Agent": "Mozilla/5.0 (RAG-Research-Pipeline/1.0)"}
            )
            with urllib.request.urlopen(req, timeout=60) as resp:
                data = resp.read()
            if len(data) < 10_000:
                raise RuntimeError(f"File too small: {url}")
            dest.write_bytes(data)
            logger.info("Saved: %s  (%.1f KB)", dest.name, len(data) / 1024)
            paths.append(dest)
            time.sleep(1.0)
        except Exception as exc:
            raise RuntimeError(f"Cannot download '{url}'. Place at '{dest}'.") from exc

    return paths


In [25]:

def load_and_chunk_documents(
    pdf_paths: List[Path],
    config: HyDEConfig,
) -> List[Document]:
    """
    Load PDFs and split into chunks, preserving paper_name and page metadata.

    Chunk size (450 chars ~ 65-75 tokens) is intentionally SHORTER than the
    HYDE_MAX_TOKENS (256 tokens). The hypothetical document should be roughly
    the same length as the chunks it targets. If chunks are 300 tokens but
    hypotheticals are 100 tokens, the embedding distributions will not align
    as well. The 450-char chunk size and 256-token hypothetical are calibrated
    to produce hypotheticals of approximately the same token length as chunks.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config.CHUNK_SIZE,
        chunk_overlap=config.CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", "! ", "? ", " ", ""],
        add_start_index=True,
    )
    all_chunks: List[Document] = []

    for pdf_path in pdf_paths:
        paper_name = pdf_path.stem.replace("_", " ").title()
        try:
            pages = PyPDFLoader(str(pdf_path)).load()
        except Exception as exc:
            raise RuntimeError(f"Failed to load '{pdf_path}': {exc}") from exc

        for page in pages:
            page.metadata.update({"source": pdf_path.name, "paper_name": paper_name})

        chunks = splitter.split_documents(pages)
        logger.info("  %s -> %d pages -> %d chunks", pdf_path.name, len(pages), len(chunks))
        all_chunks.extend(chunks)

    logger.info("Total chunks: %d", len(all_chunks))
    return all_chunks


In [45]:

 # ===========================================================================
 # SECTION 4 -- MODEL AND INDEX BUILDERS
 # ===========================================================================

def build_bge_embeddings(config: HyDEConfig) -> HuggingFaceEmbeddings:
    """Initialize BGE-large-en-v1.5 bi-encoder for standard retrieval."""
    logger.info("Loading BGE bi-encoder: %s", config.BIENCODER_MODEL)
    # Newer langchain-community versions reject query_instruction as a top-level arg.
    # Keep embeddings initialization compatible across versions.
    return HuggingFaceEmbeddings(
        model_name=config.BIENCODER_MODEL,
        model_kwargs={"device": config.BIENCODER_DEVICE},
        encode_kwargs={"normalize_embeddings": True},
    )


def build_faiss_standard(
    chunks: List[Document],
    embeddings: HuggingFaceEmbeddings,
    config: HyDEConfig,
) -> FAISS:
    """
    Build or load standard FAISS index from document chunks.

    This index is used by Configs 1-4. Documents are embedded once
    using the raw BGE encoder at index time. At query time:
        - Config 1 (baseline): query is embedded directly.
        - Configs 2-4 (HyDE): hypothetical document is embedded instead.

    The SAME FAISS index serves both the baseline and HyDE configs,
    which is the correct experimental setup: the index content is identical,
    only the QUERY REPRESENTATION changes between baseline and HyDE.
    This isolates the HyDE improvement to the query-side transformation.

    Args:
        chunks     : Document chunks.
        embeddings : BGE-large encoder (documents embedded with raw text, no instruction).
        config     : HyDEConfig.

    Returns:
        FAISS vector store with normalized inner product similarity.
    """
    faiss_file = Path(config.FAISS_STANDARD_DIR) / "index.faiss"

    if faiss_file.exists():
        logger.info("Loading standard FAISS from '%s' ...", config.FAISS_STANDARD_DIR)
        try:
            vs = FAISS.load_local(
                config.FAISS_STANDARD_DIR, embeddings,
                allow_dangerous_deserialization=True,
            )
            logger.info("Standard FAISS loaded. Vectors: %d", vs.index.ntotal)
            return vs
        except Exception as exc:
            logger.warning("FAISS load failed (%s), rebuilding ...", exc)

    logger.info("Building standard FAISS from %d chunks ...", len(chunks))
    t0 = time.perf_counter()
    vs = FAISS.from_documents(chunks, embeddings)
    elapsed = time.perf_counter() - t0
    logger.info("Standard FAISS built. Vectors: %d  (%.2fs)", vs.index.ntotal, elapsed)
    Path(config.FAISS_STANDARD_DIR).mkdir(parents=True, exist_ok=True)
    vs.save_local(config.FAISS_STANDARD_DIR)
    return vs

In [27]:

def build_faiss_hype(
    chunks: List[Document],
    embeddings: HuggingFaceEmbeddings,
    llm_hype: ChatOllama,
    config: HyDEConfig,
) -> FAISS:
    """
    Build HyPE-augmented FAISS index (index-time hypothetical question generation).

    HyPE (Hypothetical Prompt Embeddings, Vake et al. 2025) generates K
    hypothetical questions per chunk at INDEX TIME and embeds them alongside
    the original chunk. Each question embedding provides an additional entry
    point for retrieval, all pointing back to the same source chunk.

    Index structure:
        For each source chunk C_i with content text T_i:
            - Generate K questions: q_i1, q_i2, ..., q_iK that T_i answers.
            - Create K augmented documents, each with:
                  page_content = question text (the hypothetical question)
                  metadata     = original chunk metadata + {"hype_source": T_i[:500]}
            - Embed the K question texts as if they were documents.
        Add all N*K question documents to FAISS alongside the N original chunks.

    At query time, the raw query is embedded (no hypothesis generation needed)
    and matched against the question embeddings in FAISS. Because it's a
    question-to-question search (same stylistic distribution), cosine similarity
    is much higher than standard question-to-document search.

    Storage cost:
        Standard FAISS: N vectors (one per chunk)
        HyPE FAISS: N*(K+1) vectors (original + K questions per chunk)
        For N=5000 chunks, K=3 questions: 20,000 vectors total.

    Failure mode handling:
        If LLM generates < K questions for a chunk, only the available questions
        are added. If LLM fails entirely for a chunk, the original chunk is still
        added via standard embedding (zero HyPE benefit for that chunk).

    Args:
        chunks     : Source document chunks.
        embeddings : BGE-large encoder.
        llm_hype   : Azure LLM instance configured with HYPE_TEMPERATURE.
        config     : HyDEConfig.

    Returns:
        FAISS index with HyPE question embeddings + original chunk embeddings.
    """
    faiss_file = Path(config.FAISS_HYPE_DIR) / "index.faiss"

    if faiss_file.exists():
        logger.info("Loading HyPE FAISS from '%s' ...", config.FAISS_HYPE_DIR)
        try:
            vs = FAISS.load_local(
                config.FAISS_HYPE_DIR, embeddings,
                allow_dangerous_deserialization=True,
            )
            logger.info("HyPE FAISS loaded. Vectors: %d", vs.index.ntotal)
            return vs
        except Exception as exc:
            logger.warning("HyPE FAISS load failed (%s), rebuilding ...", exc)

    logger.info(
        "Building HyPE FAISS: generating %d questions per chunk for %d chunks ...",
        config.HYPE_N_QUESTIONS, len(chunks),
    )

    hype_prompt_template = """You are an expert in machine learning and NLP research.
Given the following passage from a research paper, generate exactly {n_questions} distinct,
specific questions that this passage directly answers.
Rules:
- Each question must be answerable directly and specifically from this passage alone.
- Questions should vary in focus (different aspects of the passage).
- Output ONLY the questions, one per line, no numbering, no explanation.
- Questions should be the kind a researcher would ask when searching this information.

Passage:
{passage}

{n_questions} questions:"""

    hype_prompt = ChatPromptTemplate.from_template(hype_prompt_template)

    augmented_docs: List[Document] = []

    # Add original chunks first
    augmented_docs.extend(chunks)
    logger.info("Added %d original chunks to HyPE index.", len(chunks))

    # Generate and add question embeddings for each chunk
    success_count = 0
    fail_count    = 0

    for i, chunk in enumerate(chunks):
        if i % 100 == 0:
            logger.info(
                "HyPE indexing: %d/%d chunks processed (%d questions generated, %d failed) ...",
                i, len(chunks), success_count * config.HYPE_N_QUESTIONS, fail_count,
            )
        try:
            raw_questions = (hype_prompt | llm_hype | StrOutputParser()).invoke({
                "passage":     chunk.page_content.strip()[:600],
                "n_questions": config.HYPE_N_QUESTIONS,
            })
            questions = [
                q.strip() for q in raw_questions.strip().split("\n")
                if q.strip() and len(q.strip()) > 10
            ][: config.HYPE_N_QUESTIONS]

            for q in questions:
                # Each question document points back to its source chunk
                q_doc = Document(
                    page_content=q,
                    metadata={
                        **chunk.metadata,
                        "hype_question": True,
                        "hype_source":   chunk.page_content.strip()[:500],
                    },
                )
                augmented_docs.append(q_doc)
            success_count += 1

        except Exception as exc:
            logger.debug("HyPE generation failed for chunk %d: %s", i, exc)
            fail_count += 1

    logger.info(
        "HyPE indexing complete. Total docs: %d  "
        "(original=%d, questions=%d, failed=%d chunks)",
        len(augmented_docs), len(chunks),
        len(augmented_docs) - len(chunks), fail_count,
    )

    t0 = time.perf_counter()
    vs = FAISS.from_documents(augmented_docs, embeddings)
    elapsed = time.perf_counter() - t0
    logger.info("HyPE FAISS built. Vectors: %d  (%.2fs)", vs.index.ntotal, elapsed)
    Path(config.FAISS_HYPE_DIR).mkdir(parents=True, exist_ok=True)
    vs.save_local(config.FAISS_HYPE_DIR)
    return vs



In [46]:


def build_ollama_llm(
    config: HyDEConfig,
    temperature: float = None,
) -> ChatOllama:
    """
    Initialize Ollama LLM with configurable temperature.

    Called multiple times in the pipeline:
        - temperature=0.0: answer generation (deterministic).
        - temperature=HYDE_TEMPERATURE: hypothesis generation (creative).
        - temperature=HYPE_TEMPERATURE: HyPE question generation.
    """
    missing = [n for n, v in [
        ("OLLAMA_BASE_URL", config.OLLAMA_BASE_URL),
        ("OLLAMA_MODEL", config.OLLAMA_MODEL),
    ] if not v]
    if missing:
        raise EnvironmentError(
            f"Missing Ollama configuration: {missing}\n"
            "Set OLLAMA_BASE_URL and OLLAMA_MODEL."
        )

    t = temperature if temperature is not None else config.LLM_TEMPERATURE
    logger.info(
        "Ollama: model='%s'  temperature=%.1f",
        config.OLLAMA_MODEL, t,
    )
    return ChatOllama(
        base_url=config.OLLAMA_BASE_URL,
        model=config.OLLAMA_MODEL,
        temperature=t,
        num_predict=config.LLM_MAX_TOKENS,
    )


In [29]:

# ===========================================================================
# SECTION 5 -- CORE HYDE UTILITIES
# ===========================================================================

def generate_hypothesis(
    question: str,
    llm_hyde: ChatOllama,
    prompt: str,
) -> str:
    """
    Generate a single hypothetical document for a given question.

    Args:
        question  : User query.
        llm_hyde  : LLM with HYDE_TEMPERATURE (0.5).
        prompt    : HyDE generation prompt template string with {question}.

    Returns:
        Generated hypothetical document string.
    """
    tmpl   = ChatPromptTemplate.from_template(prompt)
    result = (tmpl | llm_hyde | StrOutputParser()).invoke({"question": question})
    return result.strip()



In [30]:


def embed_hypotheticals(
    hypotheticals: List[str],
    embeddings: HuggingFaceEmbeddings,
) -> Tuple[np.ndarray, List[float]]:
    """
    Embed a list of hypothetical documents and compute the centroid.

    Each hypothetical is embedded individually using the same BGE-large encoder
    that indexed the documents. The centroid (mean vector) represents the
    average semantic direction of all hypotheses, which is used as the
    retrieval query vector in multi-hypothesis HyDE.

    Args:
        hypotheticals : List of generated hypothetical document strings.
        embeddings    : BGE-large HuggingFaceEmbeddings instance.

    Returns:
        Tuple of (centroid_vector, list_of_embedding_norms).
    """
    vectors = []
    norms   = []

    for hyp in hypotheticals:
        # embed_documents embeds as a document (no query instruction prefix)
        # This is correct: the hypothetical is a document, not a query
        vec = np.array(embeddings.embed_documents([hyp])[0], dtype=np.float32)

        norm = float(np.linalg.norm(vec))
        norms.append(norm)

        # Normalize to unit sphere for cosine similarity (FAISS IndexFlatIP)
        if norm > 1e-8:
            vec = vec / norm
        vectors.append(vec)

    centroid = np.mean(vectors, axis=0).astype(np.float32)

    # Re-normalize centroid to unit sphere (mean of unit vectors is not unit)
    centroid_norm = np.linalg.norm(centroid)
    if centroid_norm > 1e-8:
        centroid = centroid / centroid_norm

    return centroid, norms



In [31]:

def faiss_search_with_vector(
    vectorstore: FAISS,
    query_vector: np.ndarray,
    k: int,
) -> List[Document]:
    """
    Execute FAISS similarity search with a pre-computed query vector.

    This bypasses the normal embedding step in vectorstore.similarity_search(),
    allowing HyDE to use the hypothetical document embedding directly as
    the query vector rather than re-embedding the original query text.

    In LangChain's FAISS wrapper, similarity_search_by_vector() accepts
    a pre-computed embedding and performs the raw FAISS search. This is
    the correct low-level entry point for custom query vectors.

    Args:
        vectorstore  : FAISS index.
        query_vector : Pre-computed, normalized query vector (numpy float32).
        k            : Number of results to return.

    Returns:
        Top-k retrieved Documents sorted by cosine similarity (descending).
    """
    return vectorstore.similarity_search_by_vector(
        query_vector.tolist(), k=k
    )



In [32]:

def cosine_similarities(
    vec: np.ndarray,
    matrix: np.ndarray,
) -> np.ndarray:
    """
    Compute cosine similarity between a single vector and a matrix of vectors.
    Both inputs must be L2-normalized (unit norm) for inner product == cosine.

    Args:
        vec    : Query vector, shape (D,).
        matrix : Document vectors, shape (N, D), each row normalized.

    Returns:
        Cosine similarity array of shape (N,).
    """
    return matrix @ vec

In [47]:

# ===========================================================================
# SECTION 6 -- FIVE HYDE CONFIGURATIONS
# ===========================================================================

def _normalize(vec: np.ndarray) -> np.ndarray:
    norm = float(np.linalg.norm(vec))
    if norm > 1e-8:
        return (vec / norm).astype(np.float32)
    return vec.astype(np.float32)


def _answer_from_docs(
    question: str,
    docs: List[Document],
    llm_answer: ChatOllama,
    config: HyDEConfig,
) -> str:
    context = _format_docs(docs)
    prompt = ChatPromptTemplate.from_template(config.RAG_PROMPT)
    return (prompt | llm_answer | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )


def _as_source_doc(doc: Document) -> Document:
    """Convert HyPE question-doc hits back to source passage text when available."""
    source_text = doc.metadata.get("hype_source")
    if source_text:
        return Document(page_content=source_text, metadata=doc.metadata)
    return doc


def _dedupe_docs(docs: List[Document]) -> List[Document]:
    seen = set()
    unique_docs: List[Document] = []
    for doc in docs:
        key = (
            doc.metadata.get("source", ""),
            doc.metadata.get("page", ""),
            doc.page_content[:180],
        )
        if key in seen:
            continue
        seen.add(key)
        unique_docs.append(doc)
    return unique_docs


def _simple_rerank(question: str, docs: List[Document], k: int) -> List[Document]:
    """Lightweight lexical reranker to keep Config 5 dependency-free."""
    q_terms = {t for t in question.lower().split() if len(t) > 2}

    def score(doc: Document) -> float:
        text = doc.page_content.lower()
        overlap = sum(1 for t in q_terms if t in text)
        return float(overlap) + min(len(doc.page_content), 1000) / 5000.0

    ranked = sorted(docs, key=score, reverse=True)
    return ranked[:k]


def run_config1_baseline(
    question: str,
    vectorstore: FAISS,
    embeddings: HuggingFaceEmbeddings,
    llm_answer: ChatOllama,
    config: HyDEConfig,
) -> HyDETrace:
    """Configuration 1: Standard RAG baseline (no HyDE)."""
    trace = HyDETrace(question=question, strategy="Config1_Baseline_StandardRAG")
    t0 = time.perf_counter()

    t_ret = time.perf_counter()
    docs = vectorstore.similarity_search(question, k=config.SEARCH_K)
    trace.retrieval_latency_ms = (time.perf_counter() - t_ret) * 1000
    trace.retrieved_docs = docs

    trace.final_answer = _answer_from_docs(question, docs, llm_answer, config)
    trace.total_ms = (time.perf_counter() - t0) * 1000
    return trace


def run_config2_hyde_builtin(
    question: str,
    vectorstore: FAISS,
    embeddings: HuggingFaceEmbeddings,
    llm_hyde: ChatOllama,
    llm_answer: ChatOllama,
    config: HyDEConfig,
) -> HyDETrace:
    """Configuration 2: Single-hypothesis HyDE."""
    trace = HyDETrace(question=question, strategy="Config2_HyDE_SingleHypothesis")
    t0 = time.perf_counter()

    hyde_prompt = (
        "Write a concise, factual technical passage that directly answers: {question}"
    )
    t_h = time.perf_counter()
    hypothetical = generate_hypothesis(question, llm_hyde, hyde_prompt)
    trace.hyde_latency_ms = (time.perf_counter() - t_h) * 1000
    trace.hypotheticals = [hypothetical]

    t_e = time.perf_counter()
    vec = np.array(embeddings.embed_documents([hypothetical])[0], dtype=np.float32)
    vec = _normalize(vec)
    trace.hyp_embed_norms = [float(np.linalg.norm(vec))]
    trace.embed_latency_ms = (time.perf_counter() - t_e) * 1000

    t_r = time.perf_counter()
    docs = faiss_search_with_vector(vectorstore, vec, config.SEARCH_K)
    trace.retrieval_latency_ms = (time.perf_counter() - t_r) * 1000
    trace.retrieved_docs = docs

    trace.final_answer = _answer_from_docs(question, docs, llm_answer, config)
    trace.total_ms = (time.perf_counter() - t0) * 1000
    return trace


def run_config3_multi_hypothesis(
    question: str,
    vectorstore: FAISS,
    embeddings: HuggingFaceEmbeddings,
    llm_hyde: ChatOllama,
    llm_answer: ChatOllama,
    config: HyDEConfig,
) -> HyDETrace:
    """Configuration 3: Multi-hypothesis HyDE with centroid embedding."""
    trace = HyDETrace(question=question, strategy="Config3_MultiHypothesis_Centroid")
    t0 = time.perf_counter()

    hyde_prompt = (
        "Write a concise, factual technical passage that directly answers: {question}"
    )
    t_h = time.perf_counter()
    hypotheticals = [
        generate_hypothesis(question, llm_hyde, hyde_prompt)
        for _ in range(config.N_HYPOTHESES)
    ]
    trace.hyde_latency_ms = (time.perf_counter() - t_h) * 1000
    trace.hypotheticals = hypotheticals

    t_e = time.perf_counter()
    centroid, norms = embed_hypotheticals(hypotheticals, embeddings)
    trace.hyp_embed_norms = norms
    hyp_matrix = np.array(
        [_normalize(np.array(embeddings.embed_documents([h])[0], dtype=np.float32)) for h in hypotheticals],
        dtype=np.float32,
    )
    trace.centroid_cosines = cosine_similarities(centroid, hyp_matrix).tolist()
    trace.embed_latency_ms = (time.perf_counter() - t_e) * 1000

    t_r = time.perf_counter()
    docs = faiss_search_with_vector(vectorstore, centroid, config.SEARCH_K)
    trace.retrieval_latency_ms = (time.perf_counter() - t_r) * 1000
    trace.retrieved_docs = docs

    trace.final_answer = _answer_from_docs(question, docs, llm_answer, config)
    trace.total_ms = (time.perf_counter() - t0) * 1000
    return trace


def run_config4_domain_custom_prompt(
    question: str,
    vectorstore: FAISS,
    embeddings: HuggingFaceEmbeddings,
    llm_hyde: ChatOllama,
    llm_answer: ChatOllama,
    config: HyDEConfig,
) -> HyDETrace:
    """Configuration 4: Domain-specific custom prompt HyDE."""
    trace = HyDETrace(question=question, strategy="Config4_DomainSpecific_CustomPrompt")
    t0 = time.perf_counter()

    t_h = time.perf_counter()
    hypothetical = generate_hypothesis(question, llm_hyde, config.HYDE_RESEARCH_PAPER_PROMPT)
    trace.hyde_latency_ms = (time.perf_counter() - t_h) * 1000
    trace.hypotheticals = [hypothetical]

    t_e = time.perf_counter()
    vec = _normalize(np.array(embeddings.embed_documents([hypothetical])[0], dtype=np.float32))
    trace.hyp_embed_norms = [float(np.linalg.norm(vec))]
    trace.embed_latency_ms = (time.perf_counter() - t_e) * 1000

    t_r = time.perf_counter()
    docs = faiss_search_with_vector(vectorstore, vec, config.SEARCH_K)
    trace.retrieval_latency_ms = (time.perf_counter() - t_r) * 1000
    trace.retrieved_docs = docs

    trace.final_answer = _answer_from_docs(question, docs, llm_answer, config)
    trace.total_ms = (time.perf_counter() - t0) * 1000
    return trace


def run_config5_hyde_hype_reranker(
    question: str,
    vs_standard: FAISS,
    vs_hype: FAISS,
    embeddings: HuggingFaceEmbeddings,
    llm_hyde: ChatOllama,
    llm_answer: ChatOllama,
    config: HyDEConfig,
) -> HyDETrace:
    """Configuration 5: HyDE + HyPE retrieval fusion + lightweight reranking."""
    trace = HyDETrace(question=question, strategy="Config5_HyDE_HyPE_Rerank")
    t0 = time.perf_counter()

    t_h = time.perf_counter()
    hypothetical = generate_hypothesis(question, llm_hyde, config.HYDE_RESEARCH_PAPER_PROMPT)
    trace.hyde_latency_ms = (time.perf_counter() - t_h) * 1000
    trace.hypotheticals = [hypothetical]

    t_e = time.perf_counter()
    hyde_vec = _normalize(np.array(embeddings.embed_documents([hypothetical])[0], dtype=np.float32))
    trace.hyp_embed_norms = [float(np.linalg.norm(hyde_vec))]
    trace.embed_latency_ms = (time.perf_counter() - t_e) * 1000

    t_r = time.perf_counter()
    hyde_docs = faiss_search_with_vector(vs_standard, hyde_vec, config.CANDIDATE_K)
    hype_docs = [_as_source_doc(d) for d in vs_hype.similarity_search(question, k=config.CANDIDATE_K)]
    merged = _dedupe_docs(hyde_docs + hype_docs)
    reranked = _simple_rerank(question, merged, config.SEARCH_K)
    trace.retrieval_latency_ms = (time.perf_counter() - t_r) * 1000
    trace.retrieved_docs = reranked

    trace.final_answer = _answer_from_docs(question, reranked, llm_answer, config)
    trace.total_ms = (time.perf_counter() - t0) * 1000
    return trace



In [34]:
# ===========================================================================
# SECTION 7 -- SHARED FORMATTING UTILITY
# ===========================================================================

def _format_docs(docs: List[Document]) -> str:
    """Format retrieved documents into annotated context block for LLM."""
    parts = []
    for i, doc in enumerate(docs, 1):
        paper = doc.metadata.get("paper_name", "Unknown")
        page  = doc.metadata.get("page", "?")
        chars = len(doc.page_content)
        parts.append(
            f"[Rank {i} | {paper} | Page {page} | {chars} chars]\n"
            f"{doc.page_content.strip()}"
        )
    return ("\n\n" + "-" * 60 + "\n\n").join(parts)



In [35]:

# ===========================================================================
# SECTION 8 -- COMPARATIVE RUNNER
# ===========================================================================

def run_all_configs(
    question:             str,
    vs_standard:          FAISS,
    vs_hype:              FAISS,
    embeddings:           HuggingFaceEmbeddings,
    llm_hyde:             ChatOllama,
    llm_answer:           ChatOllama,
    config:               HyDEConfig,
) -> Dict[str, HyDETrace]:
    """
    Run all five HyDE configurations on a single question and print traces.
    """
    print("\n" + "#" * 78)
    print(f"QUERY: {question}")
    print("#" * 78)

    traces: Dict[str, HyDETrace] = {}

    runners = [
        ("Config1_Baseline",
         lambda q: run_config1_baseline(q, vs_standard, embeddings, llm_answer, config)),
        ("Config2_HyDE_BuiltIn",
         lambda q: run_config2_hyde_builtin(q, vs_standard, embeddings, llm_hyde, llm_answer, config)),
        ("Config3_MultiHypo",
         lambda q: run_config3_multi_hypothesis(q, vs_standard, embeddings, llm_hyde, llm_answer, config)),
        ("Config4_DomainCustomPrompt",
         lambda q: run_config4_domain_custom_prompt(q, vs_standard, embeddings, llm_hyde, llm_answer, config)),
        ("Config5_HyDE_HyPE_Reranker [BEST]",
         lambda q: run_config5_hyde_hype_reranker(q, vs_standard, vs_hype, embeddings, llm_hyde, llm_answer, config)),
    ]

    for label, fn in runners:
        print(f"\n{'='*60}\nRUNNING: {label}\n{'='*60}")
        try:
            trace = fn(question)
            trace.print_trace()
            traces[label] = trace
        except Exception as exc:
            logger.error("Config %s failed: %s", label, exc)

    # Summary table
    print("\n" + "=" * 78)
    print("HYDE LATENCY COMPARISON")
    print(f"{'Config':<44} {'HyDE(ms)':>9} {'Embed(ms)':>10} {'Ret(ms)':>8} {'Total(ms)':>10}")
    print("-" * 78)
    for lbl, tr in traces.items():
        print(
            f"{lbl:<44} {tr.hyde_latency_ms:>9.1f} "
            f"{tr.embed_latency_ms:>10.1f} "
            f"{tr.retrieval_latency_ms:>8.1f} "
            f"{tr.total_ms:>10.1f}"
        )
    print("=" * 78 + "\n")

    return traces


In [36]:
 """
    End-to-end HyDE RAG pipeline orchestrator.

    Execution order:
        1.  Download three arXiv PDFs.
        2.  Load and chunk documents.
        3.  Validate Azure credentials.
        4.  Initialize two LLM instances (temperature=0.0 and HYDE_TEMPERATURE=0.5).
        5.  Load BGE bi-encoder.
        6.  Build standard FAISS index (used by Configs 1-4).
        7.  Build HyPE FAISS index with question embeddings (used by Config 5).
            NOTE: HyPE indexing generates N_QUESTIONS*N_CHUNKS LLM calls.
            For 5000 chunks with N=3 questions: ~15,000 LLM calls (Azure batch).
            Estimated cost at GPT-4o pricing: ~$3-6 USD one-time.
            HyPE index is cached: subsequent runs skip generation.
        8.  Run all five configurations on demo queries.

    FIRST RUN TIME ESTIMATES (CPU):
        PDF download        : ~10s
        BGE embedding       : ~120s (5000 chunks)
        Standard FAISS      : ~120s (subset of above)
        HyPE indexing       : ~45min (15000 LLM calls at 400ms/call)
                              NOTE: In production, run HyPE indexing as an
                              offline batch job. The resulting index is permanent.
        Per-query (C1):     ~1.3s
        Per-query (C2-C4):  ~1.8s  (1 HyDE call + search + answer)
        Per-query (C3):     ~3.5s  (5 HyDE calls + centroid + search + answer)
        Per-query (C5):     ~2.5s  (3 HyDE + HyPE + rerank + answer)
    """

'\n   End-to-end HyDE RAG pipeline orchestrator.\n\n   Execution order:\n       1.  Download three arXiv PDFs.\n       2.  Load and chunk documents.\n       3.  Validate Azure credentials.\n       4.  Initialize two LLM instances (temperature=0.0 and HYDE_TEMPERATURE=0.5).\n       5.  Load BGE bi-encoder.\n       6.  Build standard FAISS index (used by Configs 1-4).\n       7.  Build HyPE FAISS index with question embeddings (used by Config 5).\n           NOTE: HyPE indexing generates N_QUESTIONS*N_CHUNKS LLM calls.\n           For 5000 chunks with N=3 questions: ~15,000 LLM calls (Azure batch).\n           Estimated cost at GPT-4o pricing: ~$3-6 USD one-time.\n           HyPE index is cached: subsequent runs skip generation.\n       8.  Run all five configurations on demo queries.\n\n   FIRST RUN TIME ESTIMATES (CPU):\n       PDF download        : ~10s\n       BGE embedding       : ~120s (5000 chunks)\n       Standard FAISS      : ~120s (subset of above)\n       HyPE indexing       :

In [37]:
config    = HyDEConfig()


In [38]:
logger.info("=== HyDE (Hypothetical Document Embeddings) RAG Pipeline Starting ===")


2026-05-20 21:20:34 | INFO     | hyde_rag | === HyDE (Hypothetical Document Embeddings) RAG Pipeline Starting ===


In [39]:
# Steps 1-2
pdf_paths = download_pdfs(config)
chunks    = load_and_chunk_documents(pdf_paths, config)

2026-05-20 21:20:34 | INFO     | hyde_rag | Cached: attention_is_all_you_need.pdf  (2163.3 KB)
2026-05-20 21:20:34 | INFO     | hyde_rag | Cached: bert_pretraining_transformers.pdf  (757.0 KB)
2026-05-20 21:20:34 | INFO     | hyde_rag | Cached: rag_knowledge_intensive_nlp.pdf  (864.6 KB)
2026-05-20 21:20:36 | INFO     | hyde_rag |   attention_is_all_you_need.pdf -> 15 pages -> 104 chunks
2026-05-20 21:20:36 | INFO     | hyde_rag |   bert_pretraining_transformers.pdf -> 16 pages -> 173 chunks
2026-05-20 21:20:37 | INFO     | hyde_rag |   rag_knowledge_intensive_nlp.pdf -> 19 pages -> 181 chunks
2026-05-20 21:20:37 | INFO     | hyde_rag | Total chunks: 458


In [40]:

# Steps 3-4
llm_answer = build_ollama_llm(config, temperature=config.LLM_TEMPERATURE)
llm_hyde   = build_ollama_llm(config, temperature=config.HYDE_TEMPERATURE)
llm_hype   = build_ollama_llm(config, temperature=config.HYPE_TEMPERATURE)


2026-05-20 21:21:09 | INFO     | hyde_rag | Azure OpenAI: deployment='qwen2.5-coder:7b'  temperature=0.0
2026-05-20 21:21:09 | INFO     | hyde_rag | Azure OpenAI: deployment='qwen2.5-coder:7b'  temperature=0.5
2026-05-20 21:21:09 | INFO     | hyde_rag | Azure OpenAI: deployment='qwen2.5-coder:7b'  temperature=0.4


In [48]:

# Step 5
embeddings = build_bge_embeddings(config)

2026-05-20 21:26:38 | INFO     | hyde_rag | Loading BGE bi-encoder: BAAI/bge-large-en-v1.5
2026-05-20 21:26:38 | INFO     | sentence_transformers.SentenceTransformer | Load pretrained SentenceTransformer: BAAI/bge-large-en-v1.5
2026-05-20 21:26:39 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


2026-05-20 21:26:39 | WARNING  | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-20 21:26:39 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json "HTTP/1.1 200 OK"
2026-05-20 21:26:39 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-20 21:26:39 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-20 21:26:40 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3365.28it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-05-20 21:26:43 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-20 21:26:43 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config.json "HTTP/1.1 200 OK"
2026-05-20 21:26:43 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-20 21:26:43 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-20 21:26:43 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-large-en-v1.5/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-20 21:26:44 | INFO     | httpx |

In [49]:

# Step 6: Standard FAISS (Configs 1-4)
vs_standard = build_faiss_standard(chunks, embeddings, config)


2026-05-20 21:26:49 | INFO     | hyde_rag | Building standard FAISS from 458 chunks ...
2026-05-20 21:30:34 | INFO     | faiss.loader | Loading faiss with AVX512 support.
2026-05-20 21:30:34 | INFO     | faiss.loader | Could not load library with AVX512 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx512'")
2026-05-20 21:30:34 | INFO     | faiss.loader | Loading faiss with AVX2 support.
2026-05-20 21:30:35 | INFO     | faiss.loader | Successfully loaded faiss with AVX2 support.
2026-05-20 21:30:35 | INFO     | hyde_rag | Standard FAISS built. Vectors: 458  (226.17s)


In [50]:

# Step 7: HyPE FAISS (Config 5) -- offline LLM-augmented index
vs_hype = build_faiss_hype(chunks, embeddings, llm_hype, config)


2026-05-20 21:30:50 | INFO     | hyde_rag | Building HyPE FAISS: generating 3 questions per chunk for 458 chunks ...
2026-05-20 21:30:50 | INFO     | hyde_rag | Added 458 original chunks to HyPE index.
2026-05-20 21:30:50 | INFO     | hyde_rag | HyPE indexing: 0/458 chunks processed (0 questions generated, 0 failed) ...
2026-05-20 21:31:08 | INFO     | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-20 21:31:15 | INFO     | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-20 21:31:21 | INFO     | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-20 21:31:27 | INFO     | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-20 21:31:38 | INFO     | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-20 21:31:45 | INFO     | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-20 21:31:5

In [51]:
# Step 8: Demo queries
demo_questions = [
    # Short keyword-style query -- hardest for standard dense retrieval
    "BERT pre-training objectives",

    # Interrogative query -- standard form, moderate difficulty
    "How does scaled dot-product attention compute its output?",

    # Technical comparison query -- benefits from hypothetical amplification
    "What is the difference between the encoder used in RAG and the original Transformer encoder?",

    # Precise factual query -- HyDE amplifies the answer vocabulary
    "What BLEU score did the Transformer base model achieve on WMT 2014 English-German?",

    # Cross-paper conceptual query -- broadest semantic scope
    "How do all three models handle the representation of word position in the input sequence?",
]

logger.info("Running %d queries across all 5 HyDE configurations ...", len(demo_questions))
for question in demo_questions:
    run_all_configs(question, vs_standard, vs_hype, embeddings, llm_hyde, llm_answer, config)

logger.info("=== HyDE RAG Pipeline Demo Complete ===")


2026-05-20 23:10:02 | INFO     | hyde_rag | Running 5 queries across all 5 HyDE configurations ...

##############################################################################
QUERY: BERT pre-training objectives
##############################################################################

RUNNING: Config1_Baseline
2026-05-20 23:10:15 | INFO     | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"

HYDE TRACE: Config1_Baseline_StandardRAG
Question: BERT pre-training objectives

  Retrieved Docs (4):
  [Rank 1] Bert Pretraining Transformers | Page 4 | 445 chars
            The NSP task is closely related to representation- learning objectives used in Jernite et al. (2017) and Logeswaran and Lee (2018). However, in prior ...
  [Rank 2] Bert Pretraining Transformers | Page 0 | 410 chars
            sentation models (Peters et al., 2018a; Rad- ford et al., 2018), BERT is designed to pre- train deep bidirectional representations from unlabeled text...
  [Rank 3